# A5: Seattle business licenses — coffee shop census

This notebook loads Seattle’s general business license data from **data.seattle.gov**, keeps rows whose **trade name** mentions coffee-related terms, and uses **pandas** to summarize patterns by ZIP code, opening year, and whether a brand appears to operate multiple licensed locations.

In [ ]:
import pandas as pd
# pulls the data from the seattle open portal website and limit is 50,000
URL = "https://data.seattle.gov/resource/wnbq-64tb.json?$limit=50000"

# Load as JSON (records orientation matches Socrata array-of-objects)
df = pd.read_json(URL)


## Data preview (full extract)

The following cell runs `df.head()`, `df.info()`, and `df.isnull().sum()` on the raw extract (before coffee filtering).

In [ ]:
from IPython.display import display
# preview the first 5 rows of the data 
display(df.head())
# check the columns and the names and type of data in each column 
df.info()
# checks to see the missing vales in each column, to understan what the data is missing 
df.isnull().sum()


,business_legal_name,trade_name,ownership_type,naics_code,naics_description,license_start_date,street_address,city,state,zip,business_phone,city_account_number,ubi
0,ABDURAKHMANOV RASULZHON,RASULZHON ABDURAKHMANOV,Sole proprietorship,485320,Limousine Service,20140814,706 UNION ST # 409,SEATTLE,WA,98101,2062579788,7748490686809,NaN
1,AHMED MUSTAFE F,MUSTAFE F AHMED,Sole proprietorship,485999,All Other Transit and Ground Passenger Transpo...,20141107,14203 42ND AVE S # 221,TUKWILA,WA,98168,2063849476,7760940688081,NaN
2,BUI PHUONG,BUI PHUONG,Sole proprietorship,721191,Bed-and-Breakfast Inns,20190827,2324 1ST AVE #313,SEATTLE,WA,98121,4255770134,8410320755821,6.044996e+15
3,CRUZ BRENDAN J,PHENOMENAL PHILLIE,Sole proprietorship,311811,Retail Bakeries,20220801,4130 PALATINE AVE N,SEATTLE,WA,98103,2153916626,8621920778133,6.049137e+08
4,DAVID ANTHONY LEWIS PS,DAVID ANTHONY LEWIS PS,Corporation,611699,All Other Miscellaneous Schools and Instruction,20220203,8707 16TH AVE SW,SEATTLE,WA,98106-2377,2065519336,8590290774829,6.035059e+15


<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   business_legal_name  50000 non-null  str    
 1   trade_name           50000 non-null  str    
 2   ownership_type       50000 non-null  str    
 3   naics_code           50000 non-null  int64  
 4   naics_description    49967 non-null  str    
 5   license_start_date   50000 non-null  int64  
 6   street_address       50000 non-null  str    
 7   city                 50000 non-null  str    
 8   state                49984 non-null  str    
 9   zip                  50000 non-null  str    
 10  business_phone       49729 non-null  str    
 11  city_account_number  50000 non-null  int64  
 12  ubi                  45801 non-null  float64
dtypes: float64(1), int64(3), str(9)
memory usage: 5.0 MB


business_legal_name       0
trade_name                0
ownership_type            0
naics_code                0
naics_description        33
license_start_date        0
street_address            0
city                      0
state                    16
zip                       0
business_phone          271
city_account_number       0
ubi                    4199
dtype: int64

## Build the coffee-related subset

We keep rows where `trade_name` contains **coffee**, **cafe**, or **espresso** (case-insensitive).  
For questions about Seattle, we restrict to records with `city` equal to **SEATTLE** (after normalizing case).  
ZIP codes are standardized to the first five digits when possible (handles values like `98106-2377`).

In [ ]:
# filters the data to only the businesses that have coffee, cafe or espcress in the name 
coffee_kw = r"coffee|cafe|espresso"
coffee_mask = df["trade_name"].str.contains(coffee_kw, case=False, na=False)
coffee = df.loc[coffee_mask].copy()

# filters the data to only seattle
seattle_mask = coffee["city"].fillna("").str.upper().eq("SEATTLE")
coffee_sea = coffee.loc[seattle_mask].copy()

# makes a standard so that the zip code is only 5 digits 
def clean_zip5(z):
    if pd.isna(z):
        return pd.NA
    s = str(z).split("-", 1)[0].strip()
    s = s[:5]
    return s if len(s) == 5 and s.isdigit() else pd.NA


coffee_sea["zip5"] = coffee_sea["zip"].map(clean_zip5)
coffee_sea["license_start"] = pd.to_datetime(
    coffee_sea["license_start_date"], format="%Y%m%d", errors="coerce"
)
coffee_sea["open_year"] = coffee_sea["license_start"].dt.year

trade_norm = coffee_sea["trade_name"].fillna("").str.strip().str.upper()
name_counts = trade_norm.groupby(trade_norm).transform("size")
coffee_sea["is_chain_location"] = name_counts > 1

print(f"Rows in API extract: {len(df):,}")
print(f"Coffee-related rows (all cities): {len(coffee):,}")
print(f"Coffee-related rows (Seattle only): {len(coffee_sea):,}")
coffee_sea.head()


Rows in API extract: 50,000
Coffee-related rows (all cities): 355
Coffee-related rows (Seattle only): 309


,business_legal_name,trade_name,ownership_type,naics_code,naics_description,license_start_date,street_address,city,state,zip,business_phone,city_account_number,ubi,zip5,license_start,open_year,is_chain_location
398,3156 LLC,BROADFORK CAFE,LLC - Multi Member,722511,Full-Service Restaurants,20170401,4757 12TH AVE NE,SEATTLE,WA,98105-4401,2065226966,8059050719091,6.040773e+15,98105,2017-04-01,2017,False
452,3D CAFE INC,PACIFIC CAFE,Corporation,722511,Full-Service Restaurants,20150801,416 5TH AVE S,SEATTLE,WA,98104-2806,2066820908,7897350702275,6.035252e+15,98104,2015-08-01,2015,False
494,415 COMMONS LLC,KAKAO COFFEE,Limited Liability Limited Ptrs,722515,Snack and Nonalcoholic Beverage Bars,20180101,415 WESTLAKE AVE N,SEATTLE,WA,98109,2062503576,8215480735441,6.041732e+15,98109,2018-01-01,2018,False
629,65TH ST CAFE & RESTAURANT,65TH ST CAFE & RESTAURANT,LLC - Multi Member,722511,Full-Service Restaurants,20210801,301 NE 65TH ST,SEATTLE,WA,98115-6407,2064467683,8540260769582,6.047565e+15,98115,2021-08-01,2021,False
649,70 AND SUNNY LLC,70 & SUNNY COFFEE CO,LLC - Single Member,722515,Snack and Nonalcoholic Beverage Bars,20251001,2801 ALASKAN WAY # 105,SEATTLE,WA,98121-1135,6149050517,8854750802713,6.058850e+15,98121,2025-10-01,2025,False


### 1. Which Seattle ZIP codes have the highest concentration of coffee shops?

**Definition used here:** *concentration* is measured as the **count of licensed coffee-related businesses per ZIP code** among Seattle records in this extract.  
The license file does not include population or land area, so this is a **raw density by ZIP** rather than shops per resident or per square mile.

In [ ]:
# counts the number of coffee shops licenes that exisits per zip code 
zip_counts = (
    coffee_sea.dropna(subset=["zip5"])
    .groupby("zip5", as_index=False)
    .size()
    .rename(columns={"size": "coffee_shop_licenses"})
    .sort_values("coffee_shop_licenses", ascending=False)
)







print("--- Q1: Top ZIP codes ---")
print(zip_counts.head(10))


--- Q1: Top ZIP codes ---
     zip5  coffee_shop_licenses
16  98122                    30
0   98101                    28
2   98103                    27
10  98115                    25
3   98104                    21
6   98107                    20
15  98121                    19
4   98105                    16
8   98109                    16
7   98108                    12


**Takeaway:** The table above ranks Seattle ZIPs by how many coffee-related trade names appear in the filtered extract; the top ZIPs are the most concentrated in that sense.

### 2. How has the number of new coffee shop openings changed year over year (2000–2024)?

**Definition used here:** a **new opening** is a row whose `license_start_date` parses to a calendar year between **2000** and **2024** (inclusive).  
YoY change is `openings_t - openings_{t-1}`.

In [ ]:
# counts the new coffeeshops openinfs per year to see if the is a trend in growing number of coffee shops 
openings = (
    coffee_sea.loc[coffee_sea["open_year"].between(2000, 2024)]
    .groupby("open_year")
    .size()
    .rename("new_openings")
    .sort_index()
)

yoy = openings.diff().rename("yoy_change")
openings_yoy = pd.concat([openings, yoy], axis=1)
openings_yoy


print("--- Q2: Openings by year ---")
print(openings_yoy)


--- Q2: Openings by year ---
           new_openings  yoy_change
open_year                          
2000                  1         NaN
2001                  2         1.0
2002                  6         4.0
2003                  2        -4.0
2004                  3         1.0
2005                  4         1.0
2006                  4         0.0
2007                  8         4.0
2008                  5        -3.0
2009                  4        -1.0
2010                  3        -1.0
2011                  9         6.0
2012                 10         1.0
2013                  4        -6.0
2014                 13         9.0
2015                  6        -7.0
2016                 13         7.0
2017                 13         0.0
2018                 22         9.0
2019                 20        -2.0
2020                 13        -7.0
2021                 13         0.0
2022                 17         4.0
2023                 30        13.0
2024                 25        -5.0

**Takeaway:** Positive `yoy_change` means more new licenses than the prior year; negative values mean fewer. Large swings can reflect economic cycles, licensing policy, or how businesses describe themselves in `trade_name`.

### 3. Are chains or independent coffee shops more common in Seattle?

**Definition used here (heuristic):** we normalize `trade_name` (trim + uppercase). If the **same normalized name appears on more than one Seattle row** in our coffee subset, we treat **each of those rows** as a **chain** location (multi-location brand). Rows whose normalized name appears **once** are classified as **independent** for this analysis.  
This misses franchises with slightly different spellings and can mis-label shared family names, but it is a transparent rule for license-level data.

In [ ]:
#if the trade name shows up more than once in data then it is a chain 
chain_vs_ind = (
    coffee_sea.assign(
        segment=coffee_sea["is_chain_location"].map(
            {True: "chain (brand appears 2+ times)", False: "independent (unique trade name)"}
        )
    )
    .groupby("segment", as_index=False)
    .size()
    .rename(columns={"size": "licensed_locations"})
)

chain_vs_ind["pct_of_locations"] = (
    100 * chain_vs_ind["licensed_locations"] / chain_vs_ind["licensed_locations"].sum()
).round(1)

chain_vs_ind

print("\n--- Q3: Chain vs Independent ---")
print(chain_vs_ind)


--- Q3: Chain vs Independent ---
                           segment  licensed_locations  pct_of_locations
0   chain (brand appears 2+ times)                  56              18.1
1  independent (unique trade name)                 253              81.9


**Takeaway:** Compare `licensed_locations` and `pct_of_locations` — in this extract, independents typically account for most rows because many trade names are unique.